<a href="https://colab.research.google.com/github/A-D-Vargas/PI_Mineria_Datos_1/blob/main/2_notebooks/02_calidad_y_limpieza.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Desarrollamos en 02_calidad_y_limpieza.ipynb.
Para cada decisión de preparación, indicamos en la última celda de código:
* La evidencia que la motivó
* La acción aplicada o decisión tomada con su justificación
* El impacto  o efecto observado en el dataset luego de aplicarla.

In [ ]:
import pandas as pd
import numpy as np
url = "https://raw.githubusercontent.com/A-D-Vargas/PI_Mineria_Datos_1/refs/heads/main/1_data/raw/streaming_users_dirty.json"
df_raw = pd.read_json(url)
df = df_raw.copy()

df.head()

,user_id,age,subscription_plan,monthly_watch_time_mins,country,favorite_genre,last_login_date,customer_support_tickets
0,10000,39,Estándar,805.8,Brasil,Crime,2025-03-04,99
1,10001,37,Estándar,1173.4,Colombia,Crime,2019-04-02,2
2,10002,28,Básico,401.0,Colombia,Crime,2018-04-13,0
3,10003,43,Básico,62.4,Uruguay,Thriller,2021-01-31,0
4,10004,51,Básico,477.8,Perú,Thriller,2020-09-30,1


---
#**Tratamiento de columnas numéricas**

**- Limpieza de Valores Extremos y Atípicos (Outliers)**

In [ ]:
# 1. Calculamos la mediana de edad de los usuarios sanos
mediana_edad = df[(df['age'] > 0) & (df['age'] <= 100)]['age'].median()

# 2. Reemplazamos las edades imposibles por la mediana
df.loc[(df['age'] <= 0) | (df['age'] > 100), 'age'] = mediana_edad

# 3. Reemplazar minutos imposibles por null
df.loc[(df['monthly_watch_time_mins'] < 0) | (df['monthly_watch_time_mins'] >= 50000), 'monthly_watch_time_mins'] = np.nan

# 4. Reemplazar cantidad de reclamos imposibles por null
df.loc[(df['customer_support_tickets'] < 0) | (df['customer_support_tickets'] >= 99), 'customer_support_tickets'] = np.nan

# Calculamos la mediana de la columna
mediana_minutos = df['monthly_watch_time_mins'].median()
# Calculamos la mediana de reclamos
mediana_reclamos = df['customer_support_tickets'].median()

# Imputamos y asignamos a la columna
df['monthly_watch_time_mins'] = df['monthly_watch_time_mins'].fillna(mediana_minutos)

df['customer_support_tickets'] = df['customer_support_tickets'].fillna(mediana_reclamos)

df['favorite_genre'] = df['favorite_genre'].fillna("Sin Especificar")


**- Exploración y Filtro de Minutos de Visualización**

In [ ]:
df[df['monthly_watch_time_mins'] >= 0].sort_values(by='monthly_watch_time_mins')

,user_id,age,subscription_plan,monthly_watch_time_mins,country,favorite_genre,last_login_date,customer_support_tickets
3948,13948,44,Básico,0.0,Colombia,Acción,2018-04-08,2.0
32,10032,32,Básico,0.0,Brasil,Drama,2019-10-16,0.0
2545,12545,49,Básico,0.0,México,Acción,2021-03-29,0.0
3877,13877,34,Básico,0.0,Chile,Crime,08-16-2018,3.0
98,10098,29,Básico,0.0,Argentina,Crime,2025-09-05,1.0
...,...,...,...,...,...,...,...,...
3518,13518,13,Básico,4142.6,Perú,Crime,2020-07-09,0.0
1741,11741,18,Premium,4148.3,Argentina,Documental,2021-12-02,3.0
2267,12267,47,Premium,4178.3,Brasil,Acción,2023-08-22,0.0
77,10077,18,Premium,4188.7,Chile,Documental,2021-12-26,0.0


---
#**Normalización de Texto y Fechas**
**- Análisis del Formato de las Fechas**

In [ ]:
# @title
import pandas as pd

def detectar_formato_fecha(texto):
    texto = str(texto).strip()

    # Manejo de valores nulos
    if texto in ['nan', 'None', 'NAT', ''] or not texto:
        return 'Nulo / Vacío'

    # Patrón AAAA-MM-DD (ej: 2026-07-02)
    if pd.Series(texto).str.match(r'^\d{4}-\d{2}-\d{2}$').any():
        return 'AAAA-MM-DD'

    # Patrón DD/MM/AAAA (ej: 15/08/2025)
    elif pd.Series(texto).str.match(r'^\d{2}/\d{2}/\d{4}$').any():
        return 'DD/MM/AAAA'

    # Patrón DD-MM-AAAA (ej: 02-07-2026)
    elif pd.Series(texto).str.match(r'^\d{2}-\d{2}-\d{4}$').any():
        return 'DD-MM-AAAA'

    # Patrón AAAA/MM/DD (ej: 2026/01/01)
    elif pd.Series(texto).str.match(r'^\d{4}/\d{2}/\d{2}$').any():
        return 'AAAA/MM/DD'

    else:
        return 'Otro formato / No es fecha'

# Ejecutar el conteo en tu columna
conteo_formatos = df['last_login_date'].apply(detectar_formato_fecha).value_counts()

print("--- Distribución de formatos de fecha ---")
print(conteo_formatos)

--- Distribución de formatos de fecha ---
last_login_date
AAAA-MM-DD                    7428
Nulo / Vacío                   320
DD-MM-AAAA                     265
AAAA/MM/DD                     133
Otro formato / No es fecha      14
Name: count, dtype: int64


**- Cálculo del Porcentaje de Fechas Faltantes**

In [ ]:
faltantes = df['last_login_date'].isnull().sum()
porcentaje = (faltantes / len(df['last_login_date']) * 100).round(2)
print(porcentaje)

3.92


**- Estandarización de Fechas (Formato Único)**

In [ ]:
# @title
import pandas as pd

# Creamos una función para limpiar y estandarizar cada celda individualmente
def limpiar_fecha(val):
    # Convertimos a string y quitamos espacios en blanco
    val_str = str(val).strip()

    # 1. Descartamos explícitamente los textos inválidos conocidos
    if val_str in ['not_available', '0000-00-00', 'nan', 'None']:
        return pd.NaT

    # 2. Intentamos parsear formato Año-Mes-Día (AAAA-MM-DD) o Año-Día-Mes (AAAA-DD-MM)
    if len(val_str) >= 10 and val_str[:4].isdigit() and val_str[4] == '-':
        partes = val_str.split('-')
        if len(partes) == 3:
            anio = int(partes[0])
            p1 = int(partes[1])
            p2 = int(partes[2])

            # Caso como '2026-15-03': el del medio es mayor a 12, por ende es el DÍA (AAAA-DD-MM)
            if p1 > 12 and p2 <= 12:
                try: return pd.Timestamp(year=anio, month=p2, day=p1)
                except ValueError: return pd.NaT
            # Caso estándar o ambiguo: asumimos el del medio es MES (AAAA-MM-DD)
            else:
                try: return pd.Timestamp(year=anio, month=p1, day=p2)
                except ValueError: return pd.NaT

    # 3. Intentamos parsear formato Día-Mes-Año (DD-MM-AAAA) o Mes-Día-Año (MM-DD-AAAA)
    if len(val_str) >= 10 and val_str[-4:].isdigit() and val_str[2] == '-':
        partes = val_str.split('-')
        if len(partes) == 3:
            p1 = int(partes[0])
            p2 = int(partes[1])
            a = int(partes[2])

            # Caso formato EEUU (ej: 11-19-2018): el del medio es mayor a 12, por ende es el DÍA
            if p2 > 12 and p1 <= 12:
                try: return pd.Timestamp(year=a, month=p1, day=p2)
                except ValueError: return pd.NaT
            # Caso estándar: asumimos el primero es el DÍA (DD-MM-AAAA)
            else:
                try: return pd.Timestamp(year=a, month=p2, day=p1)
                except ValueError: return pd.NaT

    # 4. Intento genérico de Pandas por si hay barras '/' o formatos ocultos
    try:
        return pd.to_datetime(val_str, errors='coerce')
    except:
        return pd.NaT

# --- APLICAMOS LA FUNCIÓN ---

# 1. Aplicamos la lógica fila por fila para rescatar todo lo real en objetos datetime nativos
df['last_login_date'] = df['last_login_date'].apply(limpiar_fecha)

# 2. Volcamos todas las fechas al formato visual unificado (AAAA-MM-DD)
df['last_login_date'] = df['last_login_date'].dt.strftime('%Y-%m-%d')

# Verificamos el resultado
print(df['last_login_date'].value_counts(dropna=False).head(20))

last_login_date
NaN           364
2026-03-15     20
2029-01-01     15
2019-01-17     10
2018-08-18      9
2025-09-29      9
2024-08-29      9
2023-02-01      8
2023-02-06      8
2018-07-27      8
2022-05-29      8
2023-09-02      8
2025-05-05      8
2024-12-27      8
2019-09-24      8
2018-11-20      8
2023-02-02      8
2025-03-20      8
2022-04-12      8
2025-11-22      8
Name: count, dtype: int64


**- Tratamiento de Fechas Nulas Restantes**

In [ ]:
df['last_login_date'] = df['last_login_date'].fillna("Desconocido")

**- Homogeneización de Columnas de Texto Categorizadas**

In [ ]:
# 1. Tratamiento preventivo para todas las columnas de texto (Pasa a minúsculas y quita espacios ocultos)
for col in ['subscription_plan', 'country', 'favorite_genre']:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip().str.lower()

# 2. Diccionarios de mapeo exhaustivos para unificar idiomas, tildes y variantes
mapa_planes = {
    'basic': 'Básico', 'básico': 'Básico', 'basico': 'Básico',
    'standard': 'Estándar', 'estándar': 'Estándar', 'estandar': 'Estándar', 'std': 'Estándar',
    'premium': 'Premium', 'premiun': 'Premium'
}
mapa_paises = {'brasil': 'Brasil', 'brazil': 'Brasil', 'bra': 'Brasil', 'uruguay': 'Uruguay', 'ury': 'Uruguay', 'argentina': 'Argentina', 'arg': 'Argentina', 'mexico': 'México', 'méxico': 'México', 'mex': 'México', 'colombia': 'Colombia', 'col': 'Colombia', 'chile': 'Chile', 'chl': 'Chile', 'peru': 'Perú', 'perú': 'Perú', 'per': 'Perú'}
mapa_generos = {
    'crimen': 'Crimen', 'crime': 'Crimen',
    'drama': 'Drama',
    'comedia': 'Comedia', 'comedy': 'Comedia',
    'acción': 'Acción', 'action': 'Acción', 'accion': 'Acción',
    'documental': 'Documental', 'documentary': 'Documental', 'doc': 'Documental',
    'romance': 'Romance',
    'thriller': 'Thriller', 'thriler': 'Thriller',
    'none': 'Sin Especificar', 'sin especificar': 'Sin Especificar'
}

# Aplicamos los reemplazos homogeneizados
df['subscription_plan'] = df['subscription_plan'].replace(mapa_planes)
df['country'] = df['country'].replace(mapa_paises)
df['favorite_genre'] = df['favorite_genre'].replace(mapa_generos)

**- Verificación de Categorías Únicas Limpias**

In [ ]:
df['country'].unique()

array(['Brasil', 'Colombia', 'Uruguay', 'Perú', 'Chile', 'Argentina',
       'México'], dtype=object)

In [ ]:
df['subscription_plan'].unique()

array(['Estándar', 'Básico', 'Premium'], dtype=object)

In [ ]:
df['favorite_genre'].unique()

array(['Crimen', 'Thriller', 'Drama', 'Acción', 'Romance', 'Comedia',
       'Documental', 'Sin Especificar'], dtype=object)

---
#**Eliminación de Duplicados, Análisis Estadístico Final y Exportación**

**-Detección de Filas Duplicadas**

In [ ]:
duplicados = df.duplicated().sum()
print(f"Número de filas duplicadas: {duplicados}")

# Filtramos las duplicadas y las ordenamos por user_id para ver las parejas/grupos juntos
f_duplicadas_ordenadas = df[df.duplicated(keep=False)].sort_values(by='user_id')

# Mostramos las primeras 10 para verificar
f_duplicadas_ordenadas.head(30)

Número de filas duplicadas: 142


,user_id,age,subscription_plan,monthly_watch_time_mins,country,favorite_genre,last_login_date,customer_support_tickets
37,10037,33,Básico,611.0,Colombia,Documental,2019-09-29,2.0
8133,10037,33,Básico,611.0,Colombia,Documental,2019-09-29,2.0
52,10052,25,Básico,465.7,Colombia,Acción,2022-03-31,1.0
8089,10052,25,Básico,465.7,Colombia,Acción,2022-03-31,1.0
8010,10117,29,Estándar,713.9,Brasil,Documental,2020-12-20,1.0
117,10117,29,Estándar,713.9,Brasil,Documental,2020-12-20,1.0
128,10128,19,Básico,638.7,Argentina,Drama,2020-06-17,1.0
8085,10128,19,Básico,638.7,Argentina,Drama,2020-06-17,1.0
8000,10156,43,Básico,592.8,Brasil,Romance,2021-10-25,0.0
156,10156,43,Básico,592.8,Brasil,Romance,2021-10-25,0.0


**-Eliminación Definitiva de Registros Duplicados**

In [ ]:
# Modifica el DataFrame actual eliminando los duplicados y dejando uno solo
df.drop_duplicates(inplace=True)

**-Auditoría de Usuarios Menores de Edad**

In [ ]:
menores_28 = df[df['age'] < 18].shape[0]
print(f"Número de usuarios menores de 28 años: {menores_28}")

Número de usuarios menores de 28 años: 739


**-Verificación del Estado Final de la Información**

In [ ]:
df.info()
df.describe()

<class 'pandas.core.frame.DataFrame'>
Index: 8018 entries, 0 to 8154
Data columns (total 8 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   user_id                   8018 non-null   int64  
 1   age                       8018 non-null   int64  
 2   subscription_plan         8018 non-null   object 
 3   monthly_watch_time_mins   8018 non-null   float64
 4   country                   8018 non-null   object 
 5   favorite_genre            8018 non-null   object 
 6   last_login_date           8018 non-null   object 
 7   customer_support_tickets  8018 non-null   float64
dtypes: float64(2), int64(2), object(4)
memory usage: 563.8+ KB


,user_id,age,monthly_watch_time_mins,customer_support_tickets
count,8018.000000,8018.000000,8018.000000,8018.000000
mean,13998.306685,33.615116,793.788401,0.803692
std,2309.066823,11.560268,487.492414,0.893828
min,10000.000000,4.000000,0.000000,0.000000
25%,11998.250000,25.000000,499.375000,0.000000
50%,13997.500000,33.000000,758.700000,1.000000
75%,15996.750000,41.000000,1029.225000,1.000000
max,17999.000000,80.000000,4193.700000,5.000000


**-Exportación a Archivo Local (CSV)**

In [ ]:
ruta_guardado_csv = '/content/streaming_users_limpio.csv'
df.to_csv(ruta_guardado_csv, index=False)
print(f"El DataFrame se ha guardado exitosamente como CSV en: {ruta_guardado_csv}")

El DataFrame se ha guardado exitosamente como CSV en: /content/streaming_users_limpio.csv


**Evidencia**:  

* **age**: en esta columna se observo que habia edades imposibles como -5, 130 y 150 años.  
* **monthly_watch_time_mins**: en esta columna se observó que habia valores imposibles porque excedian los minutos correspondientes a 1 mes (50000, 99999) o habia minutos invalidos (-1, -120).
* **customer_support_tickets**: en esta columna se observo que habia valores imposibles como -1, 99 y 150.
* **last_login_date**: en esta columna se habia encontrado valores nulos y fechas en formatos diferentes como AAAA-MM-DD, DD-MM-AAAA, AAAA/MM/DD y No es fecha.
* **subscription_plan**: en esta columna habia valores inconsistentes.
* **country**: en esta columna habia valores inconsistentes.
* **favorite_genre**: en esta columna habia valores nulos e inconsistencias.

**Acción**:

* **age**: primero detectamos y aíslamos las edades biológicamente imposibles o erróneas (valores menores o iguales a 0, o mayores a 100 años) y las reemplazamos por la mediana de las edades posibles. Se eligió usar la mediana porque es inmune a los valores extremos mantiendose firme en el centro real de todas las edades.
* **monthly_watch_time_mins**:aislamos los registros con tiempos de reproducción imposibles (valores menores a 0 o mayores/iguales a 50,000 minutos). La exclusión por debajo de cero se debe a que el tiempo negativo no existe, mientras que el tope de 50,000 se establece porque un mes promedio solo dispone de 43,200 minutos reales, lo que delata un error técnico de registro. A estas celdas erróneas primero las marcamos como vacías (np.nan) y luego las salvamos imputando con la mediana del resto de los usuarios; se prefiere esta métrica porque es la más segura para reflejar el hábito de consumo de un cliente estándar, evitando que los usuarios reales que hacen maratones extremas (valores atípicos legítimos) alteren el reemplazo.
* **customer_support_tickets** : identificamos y excluímos los valores imposibles de las celdas de reclamos. Luego las marcamos como vacías (np.nan) y las reemplazamos con la mediana de reclamos del resto de los usuarios. Usamos mediana porque el número de quejas suele estar muy concentrado en valores bajos (0 o 1 ticket por usuario), y existen pocos clientes que reclaman muchísimo (valores atípicos), y usar el promedio distorcionaria el resultado.
* **favorite_genre**: reemplazamos todas las celdas vacias o nulas por la etiqueta de texto "Sin Especificar". Porque el género es una variable categórica (de texto) y no una numérica; por lo tanto, no se le puede calcular una mediana o un promedio para rellenarla.
* **last_login_date**: estandarizamos la fecha en un formato único,
 porque el mismo estaba en formas distintas (algunas con el año al principio, otras al final, y otras con errores)

* **subscription_plan**: eliminamos los espacios en blanco, convertimos todo a minúsculas y aplicamos un diccionario de traducción que intercepta y reemplaza todas las variantes, abreviaciones y errores ortográficos registrados (como 'std', 'standard' o 'premiun').
* **country**: limpiamos los espacios en blanco, pasamos los nombres de los países a minúsculas y aplicamos un diccionario de traducción que unifica códigos internacionales de tres letras (como 'arg', 'bra', 'ury'), nombres en inglés ('brazil') y variantes con o sin tildes ('mexico', 'peru').


**Impacto observado**:

* **age** la base de datos quedó lista para segmentar correctamente a los clientes por rango de edad (sabiendo que no hay niños de 0 años ni ancianos ficticios), asegurando que los reportes reflejen al público real.

* **monthly_watch_time_mins**:no hay valores imposibles, ni nulos

*  **customer_support_tickets** veremos una columna 100% limpia de valores nulos o números incoherentes.
* **favorite_genre** no tiene ningún valor nulo. En su lugar, hay una nueva categoría llamada "Sin Especificar" que agrupa perfectamente a todos los usuarios que no eligieron un género, permitiendonos crear gráficos de barras completos donde este grupo de usuarios no desaparece del mapa, sino que se visualiza de forma honesta y ordenada.
* **last_login_date**: obtuvimos fechas reales en un formato identico Año-Mes-Día (AAAA-MM-DD).
* **subscription_plan**: se visualizan exactamente 3 categorías oficiales, limpias y correctamente escritas (Básico, Estándar, Premium). Lo que nos permitira generar gráficos exactos y métricas reales sin que los datos se dispersen en barras duplicadas.
* **country**: se visualizan las variantes en nombres de países únicos, limpios y correctamente acentuados en español (como Brasil, Argentina, México, Perú).
